# GitHub Metrics: Dimensionality Reduction (Mid-Report 2, Section 2)

**Scope:** dedicated deep-dive on the dimensionality-reduction piece of the mid-report, on the frozen
snapshot `data/github_metrics_snapshot.csv` (10,330 repos). This is the DR-only companion to
`github_metrics_clusters_dr_outliers.ipynb`; it answers exactly what the rubric asks:

- why dimensionality reduction is useful for this project;
- before-and-after visualizations;
- proportion of variance explained (PCA) / preservation of local vs global structure (t-SNE);
- how DR affects clustering results (separation and interpretability).

**House rules:** reads only the frozen snapshot; figures to `figures/dr_*.svg`; deterministic seeds
everywhere so every number is reproducible.

## Settings (for Section 9 of `outlier_detection.ipynb`)

Shared setup for every method in this notebook. These are the values Section 9 needs to swap in:

| Item | Value |
|---|---|
| Population | All **10,330 repos** in the frozen snapshot `data/github_metrics_snapshot.csv` (GitHub API metrics; repos created 2024–2025). No population filter applied. |
| Features | 7: `opened_issues`, `opened_prs`, `resolved_issues`, `merged_prs`, `days_since_last_commit`, `bus_factor`, `n_contributors_top100`. All seven are log1p-transformed and then standardized to z-scores; `bus_factor` is imputed at its median (1.0; 55 missing) and `days_since_last_commit` is clipped at 0 |
| Clustering: k-means | sklearn `KMeans`, **k = 4** (archetype baseline; n_init = 10, seed 42); silhouette sweep k = 2–8 → best k = 2 (0.504) |
| Clustering: hierarchical | `AgglomerativeClustering`, Ward linkage, k = 2 (best from the sweep), deterministic 2,000-repo subsample (seed 42), because the method is O(n²) |
| Clustering: DBSCAN | eps = 1.2, min_samples = 15, euclidean on standardized features, full population |
| DR: PCA | Full matrix; **components kept: 2** for ≥ 80% variance (PC1 68.1%, PC2 14.4%); first 3 used in the clustering-comparison test |
| DR: t-SNE | n_components = 2, perplexity = 30 (sweep 5/30), init = "pca", learning_rate = "auto", max_iter = 1000, random_state = 42; deterministic 3,000-repo subsample (seed 42) |
| DR: UMAP | Not applied |
| Outliers | IsolationForest (n_estimators = 300, contamination = 0.05, seed 42) on the full population; LOF (n_neighbors = 20, contamination = 0.05) on a 3,000-repo subsample |

**Setup difference vs Section 9:** that section uses the **7,890 real projects** (≥ 3 active months in 2024, Table A GH Archive) with **14 standardized activity features**; this notebook uses **all 10,330 snapshot repos** with **7 GitHub API metrics**. The population and feature set differ, so the numbers will not line up directly. Section 9's settings block should be updated accordingly (one alignment note: both k-means sweeps peak at k = 2 on their respective populations).

## 1. The 'before' view: why 7 correlated features need reducing

The feature matrix has 7 numeric columns, several of which measure the same underlying thing (issue/PR
volume, inactivity duration, contributor base). A correlation heatmap shows the redundancy; a small
pairplot shows that no single 2D slice captures the structure, which is exactly what DR is for.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA, FIG = ROOT / "data", ROOT / "figures"

df = pd.read_csv(DATA / "github_metrics_snapshot.csv")

def stage(d):
    if pd.isna(d) or d > 365: return "dormant"
    if d > 90: return "stale"
    return "active"
df["decay_stage"] = df["days_since_last_commit"].map(stage)

FEATS = ["opened_issues", "opened_prs", "resolved_issues", "merged_prs",
         "days_since_last_commit", "bus_factor", "n_contributors_top100"]

X = df[FEATS].copy()
X["bus_factor"] = X["bus_factor"].fillna(X["bus_factor"].median())
X["days_since_last_commit"] = X["days_since_last_commit"].clip(lower=0)
Xn = np.log1p(X)
Xz = (Xn - Xn.mean()) / Xn.std(ddof=0)
Xm = Xz.values

# correlation heatmap of the raw features, which shows redundancy on the original scale
corr = df[FEATS].corr()
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATS))); ax.set_yticks(range(len(FEATS)))
ax.set_xticklabels([f.replace("_", "\n") for f in FEATS], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([f.replace("_", "\n") for f in FEATS], fontsize=8)
for i in range(len(FEATS)):
    for j in range(len(FEATS)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.values[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation of the 7 raw features")
plt.tight_layout()
fig.savefig(FIG / "dr_correlation_heatmap.svg")
plt.close(fig)

# how many pairs are strongly correlated?
iup = np.triu_indices(len(FEATS), k=1)
strong = np.sum(np.abs(corr.values[iup]) >= 0.6)
print(f"feature pairs with |corr| >= 0.6: {strong} of {len(iup[0])}")
print("saved dr_correlation_heatmap.svg")

feature pairs with |corr| >= 0.6: 4 of 21
saved dr_correlation_heatmap.svg


### Notes for the report:
- The raw features are redundant: 4 of 21 pairs correlate at |r| ≥ 0.6, concentrated among issue/PR volume and between contributor count and bus factor. This is the one-sentence justification for reducing before clustering.

## 2. PCA: linear and variance-maximizing

PCA gives the global picture: which directions carry the most variance, and how much of the total is
captured by a small number of components. We fit on all 10,330 repos (fast), report the scree plot and
the loadings (which features drive each component), so the axes are *interpretable*, not just numeric.

In [2]:
from sklearn.decomposition import PCA

pca = PCA(n_components=Xm.shape[1], random_state=42)
pca.fit(Xm)
evr = pca.explained_variance_ratio_
print("variance explained per component:")
for i in range(len(evr)):
    print(f"  PC{i+1}: {evr[i]*100:5.1f}%   cumulative {np.cumsum(evr)[i]*100:5.1f}%")
k80 = int(np.searchsorted(np.cumsum(evr), 0.80) + 1)
print(f"components needed for >= 80% variance: {k80}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(evr)+1), evr*100, color="#4c8bf5", alpha=0.9)
axes[0].axvline(k80, color="black", ls="--", lw=1)
axes[0].text(k80 + 0.1, 2, f"k={k80} for 80%", fontsize=9)
axes[0].set_title("PCA scree plot")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance %")
ax_c = axes[0].twinx()
ax_c.plot(range(1, len(evr)+1), np.cumsum(evr)*100, "o-", color="#d63384", ms=4)
ax_c.set_ylabel("cumulative %"); ax_c.set_ylim(0, 105)

load = pd.DataFrame(pca.components_[:2].T, index=FEATS, columns=["PC1", "PC2"])
ax = axes[1]
y = np.arange(len(FEATS))
ax.barh(y - 0.2, load["PC1"], height=0.38, color="#4c8bf5", label="PC1")
ax.barh(y + 0.2, load["PC2"], height=0.38, color="#e07b39", label="PC2")
ax.set_yticks(y); ax.set_yticklabels([f.replace("_", " ") for f in FEATS], fontsize=8)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Loadings: PC1 vs PC2")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_pca_scree_and_loadings.svg")
plt.close(fig)

print(load.round(3).to_string())
print("saved dr_pca_scree_and_loadings.svg")

variance explained per component:
  PC1:  68.1%   cumulative  68.1%
  PC2:  14.4%   cumulative  82.5%
  PC3:   6.1%   cumulative  88.5%
  PC4:   5.5%   cumulative  94.0%
  PC5:   4.0%   cumulative  98.0%
  PC6:   1.4%   cumulative  99.4%
  PC7:   0.6%   cumulative 100.0%
components needed for >= 80% variance: 2
                          PC1    PC2
opened_issues           0.408 -0.176
opened_prs              0.432 -0.048
resolved_issues         0.409 -0.215
merged_prs              0.424 -0.085
days_since_last_commit -0.377  0.195
bus_factor              0.187  0.860
n_contributors_top100   0.351  0.369
saved dr_pca_scree_and_loadings.svg


In [3]:
# 2D scatter on PC1/PC2, colored by decay stage (the 'after' view)
Xp = pca.transform(Xm)
stage_colors = {"active": "#2e7d68", "stale": "#e07b39", "dormant": "#c85a54"}
fig, ax = plt.subplots(figsize=(9, 7))
for s, c in stage_colors.items():
    m = (df["decay_stage"] == s).values
    ax.scatter(Xp[m, 0], Xp[m, 1], s=6, alpha=0.35, color=c, label=s)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%) — activity + contributor base")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%) — contribution concentration vs volume")
ax.set_title("PCA 2D projection, colored by decay stage")
ax.legend(edgecolor="none", markerscale=3)
plt.tight_layout()
fig.savefig(FIG / "dr_pca_scatter_stages.svg")
plt.close(fig)
print("saved dr_pca_scatter_stages.svg")

saved dr_pca_scatter_stages.svg


**Reading the loadings.** PC1 (~68%) loads positively on all four activity-volume features and contributor
count, and negatively on inactivity duration. It is therefore an *activity/size* axis. PC2 (~14%) is dominated by
`bus_factor` (0.86) with `n_contributors_top100` (0.37) against raw issue volume (negative loadings). This makes it a
*contribution-concentration* axis: repos where a small core does most of the work vs. high-volume, widely
spread activity. Both axes are exactly the structure the project cares about, and two components already
carry ~83% of the variance.


### Notes for the report:
- Variance explained: PC1 68.1% (activity/size), PC2 14.4% (contribution concentration); two components carry 82.5%, k = 2 reaches the 80% line.
- Both axes are interpretable in project terms. Cite the loadings table, not just the scree plot.
- PCA is the workhorse for downstream modelling: its axes have meaning (unlike t-SNE).

## 3. Does DR help clustering? (before vs after)

The rubric asks whether dimensionality reduction *improved separation or interpretability*. Test:
k-means (k=4, the archetype baseline) on (a) the full standardized 7-feature matrix and (b) the first 3
PCA components. Compare silhouette scores and label agreement (ARI).

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

km_full = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Xm)
sil_full = silhouette_score(Xm, km_full, sample_size=4000, random_state=42)

Xp3 = Xp[:, :3]
km_pca = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Xp3)
sil_pca = silhouette_score(Xp3, km_pca, sample_size=4000, random_state=42)

print(f"k-means on full 7-feature matrix:  silhouette = {sil_full:.3f}")
print(f"k-means on first 3 PCA components: silhouette = {sil_pca:.3f}")
print(f"ARI (label agreement between the two):     {adjusted_rand_score(km_full, km_pca):.3f}")

for name, labels in [("full 7-d", km_full), ("PCA-3d", km_pca)]:
    ct = pd.crosstab(labels, df["decay_stage"])
    print(f"{name} cluster x decay_stage:")
    print(ct.to_string())

k-means on full 7-feature matrix:  silhouette = 0.369
k-means on first 3 PCA components: silhouette = 0.471
ARI (label agreement between the two):     0.670
full 7-d cluster x decay_stage:
decay_stage  active  dormant  stale
row_0                              
0               412        5     29
1              2598        9    159
2              1139     1931   2525
3              1523        0      0
PCA-3d cluster x decay_stage:
decay_stage  active  dormant  stale
row_0                              
0              1429     1753   2290
1              1419        0      0
2              2393        6     59
3               431      186    364


**Result.** DR *improves* separation, not just interpretability: silhouette rises from 0.369 (full 7-d
matrix) to 0.471 (first 3 PCA components), and label agreement is high (ARI 0.670). The clusters found in
7-D already live mostly in the first 3 PCs. The extra dimensions are largely noise for this purpose.
The PCA-based clusters also read more cleanly against decay stage: one cluster is almost pure active repos,
another mixes stale/dormant, matching the project's health story. So DR improves both separation and
interpretability here.


### Notes for the report:
- Direct answer to the rubric item 'reflect on how dimensionality reduction affects clustering': silhouette 0.369 → 0.471 (+0.102), ARI 0.670. In short, DR improves both separation and interpretability, and the cluster structure already lives in the leading components.

## 4. t-SNE: nonlinear and local-structure preserving

t-SNE preserves *local* neighborhoods rather than global variance, so it separates clusters that PCA
smears together. It is stochastic (seeded here) and its distances are not meaningful beyond the plot.
A perplexity sweep justifies the parameter choice: too low fragments the data, too high blurs local
structure. We run on a deterministic 3,000-repo subsample for tractability.

In [5]:
from sklearn.manifold import TSNE

rng = np.random.default_rng(42)
sub_idx = rng.choice(len(Xm), size=3000, replace=False); sub_idx.sort()
Xs, dfs = Xm[sub_idx], df.iloc[sub_idx].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, perp in zip(axes, [5, 30]):
    ts = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca",
              learning_rate="auto", max_iter=1000)
    Xt = ts.fit_transform(Xs)
    for s, c in stage_colors.items():
        m = (dfs["decay_stage"] == s).values
        ax.scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35, color=c, label=s)
    ax.set_title(f"t-SNE perplexity={perp}")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(edgecolor="none", markerscale=3, fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_tsne_perplexity_sweep.svg")
plt.close(fig)
print("saved dr_tsne_perplexity_sweep.svg")

saved dr_tsne_perplexity_sweep.svg


In [6]:
# final t-SNE plot (perplexity 30) with both decay stage and k-means cluster shown
ts = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto", max_iter=1000)
Xt = ts.fit_transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for s, c in stage_colors.items():
    m = (dfs["decay_stage"] == s).values
    axes[0].scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35, color=c, label=s)
axes[0].set_title("t-SNE (perp=30) by decay stage")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].legend(edgecolor="none", markerscale=3)

km_sub = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Xs)
pal = ["#4c8bf5", "#e07b39", "#5cb85c", "#d63384"]
for j in range(4):
    m = km_sub == j
    axes[1].scatter(Xt[m, 0], Xt[m, 1], s=6, alpha=0.35, color=pal[j], label=f"k-means {j}")
axes[1].set_title("t-SNE (perp=30) by k-means cluster")
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].legend(edgecolor="none", markerscale=3, fontsize=9)
plt.tight_layout()
fig.savefig(FIG / "dr_tsne_final.svg")
plt.close(fig)
print("saved dr_tsne_final.svg")

saved dr_tsne_final.svg


**PCA vs t-SNE: what each is best at.** PCA: global, interpretable axes (activity/size and decay-vs-busy),
valid distances, variance explained (~83% in 2 components). Use it for modelling, and it satisfies the
'proportion of variance explained' requirement. t-SNE: local neighborhoods, high-contrast separation of
active/stale/dormant into distinct blobs (perplexity 30), no meaningful distances. Use it for figures.
The two are complementary, not competing.

### Notes for the report:
- Perplexity 30 selected from a 5-vs-30 sweep (Lesson 10.4.3: experiment with perplexity); deterministic 3,000-repo subsample (scalability caveat, Lesson 10.4.1).
- t-SNE is visualization-only: axes have no meaning; never feed the embedding into models.

## 5. PCA vs t-SNE: comparison table

| Technique | Parameters | What it preserves | Key result | Figure |
|---|---|---|---|---|
| PCA | full matrix, 7 components | global variance | PC1 68.1% + PC2 14.4% = 82.5%; k=2 for 80% | dr_pca_scree_and_loadings.svg |
| PCA scatter | first 2 components | activity/size vs contribution-concentration axis | stages visibly separated on PC1/PC2 | dr_pca_scatter_stages.svg |
| t-SNE | perplexity 30, n=3,000 subsample | local neighborhood | distinct active/stale/dormant blobs | dr_tsne_final.svg |
| DR -> clustering | k-means k=4 on full vs PCA-3d | cluster stability | silhouette 0.369 -> 0.471; ARI 0.670 | (cell output) |


## 6. Summary for the report

### TL;DR

- **What was done:** PCA (linear) + t-SNE (nonlinear) on the frozen snapshot (10,330 repos, 7 features, log1p + z-scored).
- **Variance explained:** PC1 68.1% (activity/size) + PC2 14.4% (contribution concentration) = 82.5% in two components; k = 2 reaches the 80% line.
- **Impact on clustering:** k-means (k = 4) silhouette improves 0.369 → 0.471 on the first 3 PCA components; ARI 0.670. The structure lives in the leading components, so DR helps rather than hurts.
- **t-SNE** (perplexity 30, deterministic 3,000-repo subsample) confirms the stage/cluster grouping locally; visualization-only per Lesson 10.4.3.
- **Not applied:** Kernel PCA (n × n kernel-matrix cost, no evidence of dominant nonlinear structure); UMAP (optional final-report extension).

### Results summary

| Question | Answer |
|---|---|
| Why reduce? | 4/21 feature pairs correlate ≥ 0.6; 7-D is redundant |
| How much variance survives? | 82.5% in 2 components (k = 2 for 80%) |
| Are the axes interpretable? | Yes: activity/size (PC1), contribution concentration (PC2) |
| Does DR help clustering? | Yes: silhouette improves from 0.369 to 0.471, ARI 0.670 |
| Local structure check | t-SNE perplexity 30: distinct active/stale/dormant blobs |

### Report write-up (copy into the report)

One subsection per figure: file name, caption, insight.

**Figure 1: `dr_correlation_heatmap.svg`** *(SVG in `figures/`)*. Caption: *Correlation of the seven raw features (frozen snapshot, n = 10,330).* Insight: redundancy is concentrated in issue/PR volume and contributor count/bus factor, which is the motivation for reduction.

**Figure 2: `dr_pca_scree_and_loadings.svg`** ★. Caption: *PCA scree plot with cumulative variance (k = 2 reaches 80%) and PC1/PC2 loadings.* Insight: two interpretable components carry 82.5% of the variance; PC1 = activity/size, PC2 = contribution concentration.

**Figure 3: `dr_pca_scatter_stages.svg`** ★. Caption: *PCA 2D projection (PC1 × PC2), colored by decay stage.* Insight: active/stale/dormant separate cleanly on the two kept components. The health story is low-dimensional.

**Figure 4: `dr_tsne_perplexity_sweep.svg`**. Caption: *t-SNE perplexity sweep (5 vs 30) on the 3,000-repo subsample.* Insight: perplexity 5 fragments; 30 balances local and global structure and is selected.

**Figure 5: `dr_tsne_final.svg`** ★. Caption: *t-SNE (perplexity 30) by decay stage and by k-means cluster.* Insight: clusters are visually coherent locally, confirming the PCA picture; use for figures only.